### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

- A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
- A function or coroutine to execute.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Suggest asus best laptop")
response

AIMessage(content='Below is a quick‑reference guide to the **current (2024‑2025) ASUS laptops that consistently rank as the best in their categories**. I’ve grouped them by primary use‑case, listed the key specs you’ll want to look at, and added the main pros/cons plus an approximate price range (USD) as of late\u202f2024.  \n\n---\n\n## 1️⃣  All‑round “Best‑Overall” – **ASUS ZenBook\u202fPro\u202fDuo\u202f15 OLED (UX893)**  \n\n| Spec | Details |\n|------|---------|\n| **CPU** | Intel\u202fCore\u202fi9‑14900HK (12‑core) |\n| **GPU** | NVIDIA GeForce RTX\u202f4090 (Laptop, 16\u202fGB GDDR6) |\n| **Display** | 15.6‑in 4K OLED (3840×2160) 120\u202fHz + secondary 14‑in 4K ScreenPad\u202fPlus |\n| **RAM** | 32\u202fGB DDR5‑5600 (upgradeable to 64\u202fGB) |\n| **Storage** | 2\u202fTB PCIe\u202f4.0 NVMe SSD |\n| **Battery** | 94\u202fWh (≈8\u202fh mixed‑use) |\n| **Weight** | 2.5\u202fkg (5.5\u202flb) |\n| **OS** | Windows\u202f11 Pro |\n| **Price** | $2,699\u202f–\u202f$3,199 |\n\n**Why it

In [17]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather of specific location."""
    return f"Weather of {location} is sunny"

model_with_tool = model.bind_tools([get_weather])
response = model_with_tool.invoke("What the is the weather in kolkata?")
response
# for tool_call in response.tool_calls:
#     print(f"Tool : {tool_call.get("name")}")
#     print(f"args : {tool_call.get("args")}")

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What the is the weather in kolkata?" Likely a typo: "What is the weather in Kolkata?" We need to fetch weather via function get_weather. Use location "Kolkata".', 'tool_calls': [{'id': 'fc_9cbd2da4-801a-47c1-aa84-034055b47328', 'function': {'arguments': '{"location":"Kolkata"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 129, 'total_tokens': 203, 'completion_time': 0.156155012, 'completion_tokens_details': {'reasoning_tokens': 45}, 'prompt_time': 0.006309921, 'prompt_tokens_details': None, 'queue_time': 0.156183997, 'total_time': 0.162464933}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_86dc843b12', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1db43-f0cc-7256-88f9-8e8365c19b56-0', tool_calls=[{'name': 'get_weather', 'args': {'location':

#### Tool Execution Loop

In [16]:
# 1. Model generates tool call
messages = [{"role":"user","content":"What the is the weather in kolkata?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

# 2. Execute tool and collect result
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# 3. Pass result to the model to get final result
response = model_with_tool.invoke(messages)
response


AIMessage(content='The weather in Kolkata is currently sunny.', additional_kwargs={'reasoning_content': 'The user asked "What the is the weather in kolkata?" Probably typo "What the is". The assistant should respond with the weather info. The tool returned "Weather of Kolkata is sunny". So respond.'}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 161, 'total_tokens': 221, 'completion_time': 0.12603048, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.043568816, 'prompt_tokens_details': None, 'queue_time': 0.569234238, 'total_time': 0.169599296}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1dacf-c7bb-72c8-a4b0-f959cf7237c5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 161, 'output_tokens': 60, 'total_tokens': 221, 'output_token_details': {'reasoning': 43}})